In [ ]:
! pip install tensorflow numpy pillow gradio matplotlib || echo "Instalacion completa"


In [ ]:
import numpy as np
import tensorflow as tf
import gradio as gr
from PIL import Image
import matplotlib.pyplot as plt
import io


In [ ]:

(model, _), _ = (tf.keras.datasets.mnist.load_data(), None)
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='softmax')
])
# Entrenamos un poquito (solo ejemplo, omitir si ya tienes modelo)
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train[:1000].astype("float32") / 255.0
y_train = y_train[:1000]
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["acc"])
model.fit(x_train[..., None], y_train, epochs=1, verbose=0)


In [ ]:

# -----------------------------
# Función de predicción + saliency map
# -----------------------------
def predecir_digito_con_saliency(image):
    try:
        if image is None:
            return "Por favor dibuja un número 🙃", None

        # Extraer imagen del sketchpad
        if isinstance(image, dict):
            img_array = image.get("composite", None)
            if img_array is None:
                return "⚠️ No se pudo obtener la imagen del input ('composite' vacío).", None
        else:
            return f"⚠️ Tipo de entrada no soportado: {type(image)}", None

        # Preprocesamiento
        img_gray = np.array(img_array).astype(np.uint8)
        img_gray = 255 - img_gray  # Invertir colores
        img = Image.fromarray(img_gray).resize((28, 28))
        arr = np.array(img) / 255.0
        arr = arr.reshape(1, 28, 28, 1)

        # Predicción
        preds = model.predict(arr, verbose=0)
        pred_label = np.argmax(preds)
        pred_conf = np.max(preds)

        # Calcular saliency map
        with tf.GradientTape() as tape:
            tape.watch(arr)
            preds = model(arr)
            loss = preds[:, pred_label]

        grads = tape.gradient(loss, arr)
        saliency = np.abs(grads[0]).max(axis=-1)

        # Normalizar mapa [0,1]
        saliency = (saliency - saliency.min()) / (saliency.max() + 1e-8)

        # Convertir saliency map a imagen para mostrar
        plt.figure(figsize=(2.8, 2.8))
        plt.axis("off")
        plt.imshow(saliency, cmap="hot")
        buf = io.BytesIO()
        plt.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)
        plt.close()
        buf.seek(0)
        saliency_img = Image.open(buf)

        texto = f"🧠 El modelo predice: {pred_label} (confianza: {pred_conf:.2f})"
        return texto, saliency_img

    except Exception as e:
        import traceback
        return f"⚠️ Error: {str(e)}\n\n{traceback.format_exc()}", None



In [ ]:

# -----------------------------
# Interfaz Gradio
# -----------------------------
iface = gr.Interface(
    fn=predecir_digito_con_saliency,
    inputs=gr.Sketchpad(
        crop_size=(256, 256),
        type="numpy",
        image_mode="L",
        brush=gr.Brush(),
        label="Dibuja un número (0–9)"
    ),
    outputs=[
        gr.Textbox(label="Predicción del modelo"),
        gr.Image(label="Mapa de saliencia")
    ],
    title="🖍️ Dibuja un número y observa cómo piensa la red",
    description="La red convolucional predice el dígito y genera un saliency map que muestra qué partes influyeron más en su decisión."
)



In [ ]:
iface.launch(inline=True, debug=True)
